# CPSC 368 – Group 4: YouTube Data Collection
## Phase 3 – Data Collection via YouTube Data API v3

This notebook collects channel stats, video stats, and top comments for four political YouTube channels:

| Channel | Lean |
|---|---|
| Fox News | Conservative |
| Ben Shapiro / Daily Wire | Conservative |
| CNN | Liberal |
| The Young Turks | Liberal |

**Output:** Three CSV files (`channels.csv`, `videos.csv`, `comments.csv`) ready for Oracle ingestion.


## 1. Configuration

In [1]:
import time
import csv
from googleapiclient.discovery import build
import json
import pandas as pd
import os

from dotenv import load_dotenv

In [2]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [20]:
CHANNELS = [
    {"id": "UCXIJgqnII2ZOINSWNOGFThA", "name": "Fox News",          "lean": "conservative"},
    {"id": "UCnQC_G5Xsjhp9fEJKuIcrSw", "name": "Ben Shapiro",       "lean": "conservative"},
    {"id": "UCupvZG-5ko_eiXAupbDfxWw", "name": "CNN",               "lean": "liberal"},
    {"id": "UC1yBKRuGpC1tSM73A0ZjYjQ", "name": "The Young Turks",   "lean": "liberal"},
]

VIDEOS_PER_CHANNEL = 8*12*2  # 8 years, 2 videos per month = 192 videos per channel
COMMENTS_PER_VIDEO = 5      # 192 videos per channel * 5 comments per video = 960 comments per channel
START_YEAR = 2016
END_YEAR   = 2024

# output file names
OUT_CHANNELS = "../../cleaned_data/youtube/channels.csv"
OUT_VIDEOS   = "../../cleaned_data/youtube/videos.csv"
OUT_COMMENTS = "../../cleaned_data/youtube/comments.csv"

os.makedirs("../../cleaned_data/youtube", exist_ok=True)

# setup youtube api
youtube = build("youtube", "v3", developerKey=API_KEY)

## 3. Fetch Channel Stats

In [4]:
def fetch_channel_stats(channel_id, lean):
    request = youtube.channels().list(
        part="snippet,contentDetails,statistics",
        id=channel_id,
    )

    resp = request.execute()
    if not resp or not resp.get("items"):
        return None
    item  = resp["items"][0]
    stats = item.get("statistics", {})
    return {
        "channel_id":        channel_id,
        "channel_name":      item["snippet"]["title"],
        "lean":              lean,
        "description":       item["snippet"].get("description", ""),
        "published_at":      item["snippet"].get("publishedAt", ""),
        "subscriber_count":  stats.get("subscriberCount", ""),
        "total_view_count":  stats.get("viewCount", ""),
        "total_video_count": stats.get("videoCount", ""),
        "uploads_playlist":  item["contentDetails"]["relatedPlaylists"]["uploads"],
    }

# collect channel stats for all 4 channels
all_channels = []
for ch in CHANNELS:
    print(f"Fetching stats for: {ch['name']}...")
    stats = fetch_channel_stats(ch["id"], ch["lean"])
    if stats:
        all_channels.append(stats)
        print(f"  Subscribers : {stats['subscriber_count']}")
        print(f"  Total views : {stats['total_view_count']}")
        print(f"  Total videos: {stats['total_video_count']}")
    print()

print(f"Channel stats collected for {len(all_channels)} channels.")

Fetching stats for: Fox News...
  Subscribers : 15100000
  Total views : 24451907328
  Total videos: 135240

Fetching stats for: Ben Shapiro...
  Subscribers : 7090000
  Total views : 4646076563
  Total videos: 9719

Fetching stats for: CNN...
  Subscribers : 19300000
  Total views : 20147351569
  Total videos: 180748

Fetching stats for: The Young Turks...
  Subscribers : 6500000
  Total views : 7306545335
  Total videos: 71009

Channel stats collected for 4 channels.


In [ ]:
df = pd.json_normalize(all_channels)
df.to_csv(OUT_CHANNELS, index=False)
display(df)

,channel_id,channel_name,lean,description,published_at,subscriber_count,total_view_count,total_video_count,uploads_playlist
0,UCXIJgqnII2ZOINSWNOGFThA,Fox News,conservative,FOX News Channel (FNC) is a 24-hour all-encomp...,2006-09-19T01:48:52Z,15100000,24451907328,135240,UUXIJgqnII2ZOINSWNOGFThA
1,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01T00:13:29Z,7090000,4646076563,9719,UUnQC_G5Xsjhp9fEJKuIcrSw
2,UCupvZG-5ko_eiXAupbDfxWw,CNN,liberal,CNN is the world leader in news and informatio...,2005-10-02T16:06:36Z,19300000,20147351569,180748,UUupvZG-5ko_eiXAupbDfxWw
3,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21T20:46:51Z,6500000,7306545335,71009,UU1yBKRuGpC1tSM73A0ZjYjQ


## 5. Fetch Video IDs

In [ ]:
def fetch_all_videos_for_channel(channel_name, uploads_playlist_id):
    os.makedirs("checkpoints", exist_ok=True)
    checkpoint_file = f"checkpoints/{channel_name.lower().replace(' ', '_').replace('/', '')}.json"

    # checkpoint file incase we stopped running for a channel midway
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            checkpoint = json.load(f)
        video_ids   = checkpoint["videos"]
        next_page   = checkpoint["last_page_token"]
        page_num    = checkpoint["last_page_num"]
        already_seen = {v["video_id"] for v in video_ids}
        print(f"Resuming {channel_name} from page {page_num} ({len(video_ids)} videos already collected)")
    else:
        video_ids    = []
        next_page    = None
        page_num     = 0
        already_seen = set()
        print(f"Starting fresh for {channel_name}")

    done = False
    while not done:
        resp = youtube.playlistItems().list(
            part="contentDetails,snippet",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=next_page
        ).execute()

        if not resp:
            print(f"Error on page {page_num} for {channel_name}")
            break

        for item in resp.get("items", []):
            pub = item["snippet"].get("publishedAt", "")
            if not pub:
                continue
            year     = int(pub[:4])
            video_id = item["contentDetails"]["videoId"]
            video_title = item["snippet"].get("title", "")

            if year < START_YEAR:
                print(f"Reached pre-{START_YEAR} content — stopping.")
                done = True
                break

            if year > END_YEAR or video_id in already_seen:
                continue

            video_ids.append({
                "video_id":     video_id,
                "title":        video_title,
                "published_at": pub,
                "page":         page_num,
            })
            already_seen.add(video_id)

        next_page = resp.get("nextPageToken")
        page_num += 1

        # Save checkpoint after every page
        with open(checkpoint_file, "w") as f:
            json.dump({
                "channel_name":    channel_name,
                "last_page_token": next_page,
                "last_page_num":   page_num,
                "videos":          video_ids,
            }, f)

        if page_num % 10 == 0:
            print(f"Finished page {page_num}, {len(video_ids)} IDs so far. Currently at {pub}")

        if not next_page:
            break

        time.sleep(0.5)

    print(f"Found {len(video_ids)} videos in range for {channel_name}")
    return video_ids

In [ ]:
resp = youtube.playlistItems().list(
    part="contentDetails,snippet",
    playlistId=uploads_playlist_id,
    maxResults=50,
    pageToken=next_page
).execute()

### one channel at a time

In [7]:
all_channels_df = pd.read_csv(OUT_CHANNELS)
all_channels_df

,channel_id,channel_name,lean,description,published_at,subscriber_count,total_view_count,total_video_count,uploads_playlist
0,UCXIJgqnII2ZOINSWNOGFThA,Fox News,conservative,FOX News Channel (FNC) is a 24-hour all-encomp...,2006-09-19T01:48:52Z,15100000,24451907328,135240,UUXIJgqnII2ZOINSWNOGFThA
1,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01T00:13:29Z,7090000,4646076563,9719,UUnQC_G5Xsjhp9fEJKuIcrSw
2,UCupvZG-5ko_eiXAupbDfxWw,CNN,liberal,CNN is the world leader in news and informatio...,2005-10-02T16:06:36Z,19300000,20147351569,180748,UUupvZG-5ko_eiXAupbDfxWw
3,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21T20:46:51Z,6500000,7306545335,71009,UU1yBKRuGpC1tSM73A0ZjYjQ


In [17]:
# channel_name = 'Ben Shapiro'
# channel_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["channel_id"].values[0]
# uploads_playlist_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["uploads_playlist"].values[0]

# ids = fetch_all_videos_for_channel(channel_name, uploads_playlist_id)

channel_name = 'The Young Turks'
channel_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["channel_id"].values[0]
uploads_playlist_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["uploads_playlist"].values[0]

ids = fetch_all_videos_for_channel(channel_name, uploads_playlist_id)

Starting fresh for The Young Turks
Finished page 10, 0 IDs so far. Currently at 2026-01-14T05:15:02Z
Finished page 20, 0 IDs so far. Currently at 2025-11-01T20:00:21Z
Finished page 30, 0 IDs so far. Currently at 2025-09-01T22:00:29Z
Finished page 40, 0 IDs so far. Currently at 2025-07-04T20:00:25Z
Finished page 50, 0 IDs so far. Currently at 2025-05-08T02:00:29Z
Finished page 60, 0 IDs so far. Currently at 2025-03-07T05:00:02Z
Finished page 70, 10 IDs so far. Currently at 2024-12-30T03:00:00Z
Finished page 80, 510 IDs so far. Currently at 2024-10-22T00:45:02Z
Finished page 90, 1010 IDs so far. Currently at 2024-08-21T02:45:03Z
Finished page 100, 1510 IDs so far. Currently at 2024-06-25T00:00:33Z
Finished page 110, 2010 IDs so far. Currently at 2024-04-25T03:00:13Z
Finished page 120, 2510 IDs so far. Currently at 2024-03-02T05:00:28Z
Finished page 130, 3010 IDs so far. Currently at 2024-01-11T04:30:13Z
Finished page 140, 3510 IDs so far. Currently at 2023-11-24T21:00:33Z
Finished page 1

## 5.5 Showcasing Video ID DF

In [16]:
with open('checkpoints/the_young_turks.json', 'r') as f:
    tyt_data = json.load(f)

with open('checkpoints/ben_shapiro.json', 'r') as f:
    bs_data = json.load(f)

print('========= The Young Turks =========')
tyt_df = pd.DataFrame(tyt_data["videos"])
display(pd.concat([tyt_df.head(5), tyt_df.tail(5)]))

print('========= Ben Shapiro =========')
bs_df = pd.DataFrame(bs_data["videos"])
display(pd.concat([bs_df.head(5), bs_df.tail(5)]))

========= The Young Turks =========


,video_id,title,published_at,page
0,e4LwriSxOF8,TYT's 2024 Turk of the Year,2024-12-31T22:40:06Z,69
1,OHbfdKT9GGY,Judge Chides Lawyer and Police Over Ridiculous...,2024-12-31T21:30:01Z,69
2,9B4VMeHqyzc,TYT's 2024 Jerk of the Year,2024-12-31T05:45:05Z,69
3,yS_DKwj2jVU,U.S. Homelessness Rate Just Had A Serious SPIKE,2024-12-31T04:30:26Z,69
4,cM71HcAaPOA,Israel SLAMMED For SHUTTERING Last Functional ...,2024-12-31T03:45:02Z,69
10978,n7-uJC92QgY,The Young Turks 11.16.17: Al Franken Allegatio...,2017-11-17T03:43:00Z,289
10979,oLeZXeY53Rk,"The Young Turks 11.14.17: Wikileaks, Sessions,...",2017-11-15T02:38:00Z,289
10980,QJXkt_XiL68,"The Young Turks 11.13.17: Feinstein, Big Pharm...",2017-11-14T02:07:00Z,289
10981,2Qu79o3CWUY,The Young Turks 11.10.17: Roy Moore & Louis CK,2017-11-11T22:52:00Z,289
10982,j25IOnmEp0k,"The Young Turks 11.09.17: Roy Moore, Trump in ...",2017-11-10T01:51:00Z,289


========= Ben Shapiro =========


,video_id,title,published_at,page
0,KFXxLKQ1Z_E,"An important, memorable moment I got to experi...",2024-12-30T23:00:18Z,43
1,onYKY_7U2Bc,One of the BEST things that happened all year,2024-12-30T20:00:23Z,43
2,kzavxVlggFc,A Look Back At 2024 | Ben’s Best Moments,2024-12-30T18:00:34Z,43
3,D88S3niEUV8,My producers got me these HILARIOUS tiny boxin...,2024-12-30T00:00:09Z,43
4,YwhRwtszmD4,Lightsaber chopsticks have ENDLESS possibilities,2024-12-29T21:00:17Z,43
7556,g5wEep7nLOg,Ep. 208 - President-Elect Trump,2016-11-11T00:32:42Z,194
7557,gaQ8ISE5E2Y,Ep. 207 - Today Is The Day,2016-11-11T00:32:38Z,194
7558,HMYpHlxChmo,Ep. 206 - The Final Prediction: Trump vs. Clinton,2016-11-11T00:32:32Z,194
7559,g29TugDYLRw,Ep. 205 - The Final Countdown,2016-11-07T05:49:30Z,194
7560,PY09dED8404,Ep. 204 - Will Trump Provide Another Cubs-Like...,2016-11-07T05:49:26Z,194


## 6. Fetch Video Stats